# Voice Activity Detection

> Detecting where speech is - the front-end for ASR, diarization and streaming: how VAD works, the mid-2026 landscape, evaluation, and runnable code on GPU or CPU.

- skip_showdoc: true
- skip_exec: true

## 1. What is Voice Activity Detection?

Voice Activity Detection (VAD) decides, frame by frame, whether **speech is present** - separating it from silence, noise and music.

**Input.** An audio stream (usually 16 kHz mono).

**Output.** Per-frame speech probability, collapsed into **speech segments** with start/end timestamps.

**Why it matters.** VAD gates and segments audio before heavier models: trim silence before ASR, chunk long recordings, detect end-of-utterance (endpointing) in streaming, and provide the speech regions for **diarization**.

**Neighbouring tasks:** audio classification (VAD is the binary speech/no-speech case), ASR, diarization, and sound-event detection.

Note: `transformers` has no dedicated VAD pipeline; the field standard is **Silero VAD** (via `torch.hub`), with `pyannote.audio` and `speechbrain` for segmentation.

---

## 2. How Modern VAD Works

1. **Signal thresholds (classic).** Energy and zero-crossing rate; **WebRTC VAD** (a GMM, still ubiquitous) - extremely light and fast, weak in noise.
2. **DNN frame classifiers (2015+).** A small network over log-mel frames predicts speech probability.
3. **Silero VAD (2020-2024).** A tiny (~1.8M param) CNN/RNN, ONNX-deployable, millisecond latency - the **de-facto open VAD**, robust across languages and noise.
4. **Neural segmentation (pyannote, 2021-2024).** SincNet/transformer models that jointly do VAD, **overlapped-speech** detection and diarization.
5. **Self-supervised (wav2vec2 / WavLM).** Fine-tuned for segmentation when maximum robustness is needed. By 2025-2026 Silero remains the practical default; pyannote leads multi-speaker segmentation.

---

## 3. Evaluation Metrics

- **Frame-level accuracy / precision / recall / F1.** Treat each frame as a binary classification.
- **ROC-AUC.** Threshold-independent quality of the speech-probability score.
- **False-alarm and miss rates.** FA (non-speech marked speech) vs miss (speech marked non-speech) - the operating-point trade-off.
- **Detection latency.** For streaming endpointing.

The cell computes precision/recall/F1 with `scikit-learn` on toy frame labels.

---

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# frame labels: 1 = speech, 0 = non-speech
true = [0, 0, 1, 1, 1, 1, 0, 0, 1, 1]
pred = [0, 1, 1, 1, 1, 0, 0, 0, 1, 1]
p, r, f, _ = precision_recall_fscore_support(true, pred, average="binary")
print(f"precision {p:.2f}  recall {r:.2f}  F1 {f:.2f}")

## 4. The Model Landscape (mid-2026)

| Model | Params | License | Library | Best for |
|-------|--------|---------|---------|----------|
| Silero VAD | 1.8M | MIT | torch.hub | de-facto default, tiny, streaming |
| pyannote/segmentation-3.0 | 6M | MIT (gated) | pyannote.audio | VAD + overlap + diarization |
| speechbrain/vad-crdnn-libriparty | 3M | Apache 2.0 | speechbrain | CRDNN VAD |
| webrtcvad | tiny | BSD-3 | webrtcvad | ultra-light classic baseline |

There is no single VAD leaderboard; segmentation quality is reported on **AMI** and **DIHARD**. We use **Silero VAD** (general-purpose, `torch.hub`) as the primary model and **WebRTC** as a classic baseline.

---

## 5. Setup

Package roles:

- `torch` - runs Silero VAD (fetched via `torch.hub`, tiny, CPU-friendly)
- `datasets` + `soundfile` - a speech clip we pad with silence to build a labelled test signal
- `numpy` - frame handling; `scikit-learn` - frame metrics
- `pyecharts` - F1 bar + a speech-probability timeline

`webrtcvad` is an optional extra shown in prose.

---

In [ ]:
import gc
import time
import urllib.request
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:16s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "GC then release cached CPU/GPU memory. Call right after `del model`.\n\n    `del` drops the Python reference; this reclaims the RAM and hands the\n    freed VRAM back to the CUDA allocator so usage stays flat across cells.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# All downloads (samples, HF cache) go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)

import io

import numpy as np
import soundfile as sf
from datasets import Audio, load_dataset

SR = 16000

# Build a labelled test signal: [silence | speech | silence | speech | silence]
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean",
                  split="validation", cache_dir=str(DATA_DIR / "hf_cache"))
ds = ds.cast_column("audio", Audio(decode=False))


def _decode(a):
    src = a["bytes"] or a["path"]
    arr, sr = sf.read(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src, dtype="float32")
    return arr

speech = _decode(ds[0]["audio"])[: 3 * SR]  # 3 s of speech
gap = np.zeros(SR, dtype="float32")          # 1 s of silence
signal = np.concatenate([gap, speech, gap, speech, gap])

# ground-truth speech mask (1 per sample), then per-frame labels (10 ms frames)
mask = np.concatenate([np.zeros(SR), np.ones(len(speech)), np.zeros(SR),
                       np.ones(len(speech)), np.zeros(SR)])
FRAME = SR // 100  # 10 ms
frame_true = [int(mask[i:i + FRAME].mean() > 0.5) for i in range(0, len(mask) - FRAME, FRAME)]
print(f"signal {len(signal) / SR:.1f}s, {len(frame_true)} frames, "
      f"{sum(frame_true)} speech frames")

## 6. Silero VAD

Silero loads from `torch.hub` (cached under `DATA_DIR`) and returns speech segments directly via `get_speech_timestamps`. It is tiny and fast enough to run comfortably on CPU.

---

In [ ]:
torch.hub.set_dir(str(DATA_DIR / "torch_hub"))
model, utils = torch.hub.load("snakers4/silero-vad", "silero_vad", trust_repo=True)
get_speech_timestamps = utils[0]

wav = torch.from_numpy(signal)
t0 = time.perf_counter()
segments = get_speech_timestamps(wav, model, sampling_rate=SR)
elapsed = time.perf_counter() - t0
print(f"{elapsed:.3f}s  RTF {elapsed / (len(signal) / SR):.4f}")
for s in segments:
    print(f"  speech {s['start'] / SR:5.2f}s -> {s['end'] / SR:5.2f}s")

# per-frame prediction from the returned segments
frame_pred = [0] * len(frame_true)
for s in segments:
    for fi in range(s["start"] // FRAME, min(s["end"] // FRAME, len(frame_pred))):
        frame_pred[fi] = 1

del model
free_memory()
vram("after silero")

## 7. Benchmark and speech timeline

Score Silero's frames against the ground-truth mask (precision / recall / F1) and chart both the **F1** and the **speech timeline** (predicted vs true) with ECharts. For a real benchmark, run over AMI / DIHARD with the official scoring; this constructed clip is a smoke test.

---

In [ ]:
# ECharts (pyecharts) is the repo standard for all charts - it renders interactive
# and embeds straight into the Quarto docs via .render_notebook().
from pyecharts import options as opts
from pyecharts.charts import Bar


def bar_chart(title, categories, series, y_name=""):
    "Grouped bar chart. `series` is a dict {name: [values aligned to categories]}."
    chart = Bar(init_opts=opts.InitOpts(width="720px", height="420px"))
    chart.add_xaxis([str(c) for c in categories])
    for name, vals in series.items():
        chart.add_yaxis(name, [round(float(v), 4) for v in vals])
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=title),
        yaxis_opts=opts.AxisOpts(name=y_name),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=20)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
    return chart.render_notebook()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f, _ = precision_recall_fscore_support(frame_true, frame_pred, average="binary")
print(f"Silero  precision {p:.2f}  recall {r:.2f}  F1 {f:.2f}")

bar_chart("VAD frame-level scores (Silero)", ["precision", "recall", "F1"],
          {"Silero VAD": [p, r, f]}, y_name="score")

In [ ]:
# Speech timeline: true vs predicted speech over time (a step comparison)
from pyecharts import options as opts
from pyecharts.charts import Line

times = [round(i * FRAME / SR, 2) for i in range(len(frame_true))]
tl = Line(init_opts=opts.InitOpts(width="760px", height="300px"))
tl.add_xaxis(times)
tl.add_yaxis("true speech", frame_true, is_step=True, is_symbol_show=False,
             areastyle_opts=opts.AreaStyleOpts(opacity=0.2))
tl.add_yaxis("Silero predicted", [v + 0.02 for v in frame_pred], is_step=True, is_symbol_show=False)
tl.set_global_opts(
    title_opts=opts.TitleOpts(title="Speech activity over time"),
    xaxis_opts=opts.AxisOpts(name="seconds", type_="value"),
    yaxis_opts=opts.AxisOpts(name="speech (1) / silence (0)"),
    tooltip_opts=opts.TooltipOpts(trigger="axis"),
)
tl.render_notebook()

## 8. Going Further

- **Classic baseline.** `webrtcvad` (`pip install webrtcvad`) is a GMM VAD with 3-4 aggressiveness levels - great when you need a dependency-free, ultra-fast gate.
- **Diarization-grade segmentation.** `pyannote/segmentation-3.0` (gated, needs an HF token) adds **overlapped-speech** detection and feeds `pyannote`'s diarization pipeline.
- **As an ASR front-end.** Silero timestamps drive chunked long-form transcription (whisper-streaming, RealtimeSTT) - transcribe only the speech regions.
- **Streaming endpointing.** Silero runs frame-by-frame with a few-ms lookahead for live end-of-utterance detection.
- **Real evaluation.** Score against AMI / DIHARD references with `pyannote.metrics` (detection error rate, FA / miss).

---